# FSCI 396: Structural Equation Modelling

------------------
Welcome to today's tutorial! 
Today we'll use Python in Jupyter Notebook to perform some common statistical analyses.
We will cover: Structural Equation Modelling

The goal is to see how we can analyze data with Python, just like we discussed in lectures.


### Steps to Debugging 

I am here to help you! But before you contact me, make sure you have tried the following:
1. Ensure that all brackets and parentheses are paired.
2. Ensure that your code does not have any typos (eg. when calling your data file).
3. Ensure you did not add additional spaces (eg. when calling your data file).
4. You have restarted your kernel and reun your code. 
5. You have tried to understand the error message. 

If you have done all of these things, I am very happy to help! 

In [ ]:
# Import Modules
# ---------------

# Modules in Python are like library books. 
# Each book contains a set of instructions or “recipes” for doing specific tasks.
# - pandas: for handling and analyzing tables of data
# - semopy: used for structural equation modelling 

import pandas as pd
import pingouin as pg
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


# Ignore Warnings for simplicity
import warnings
warnings.filterwarnings('ignore')

# Quick check to see if modules loaded correctly
print("Modules loaded successfully!")


In [ ]:
# Load datafile/dataframe (df) from CSV
# --------------------------------------

# Here we load our data. Make sure the CSV file is in the same folder as this notebook.

# If your data is .csv, use these two lines: 
filename = 'SampleData.csv'    # Name of your file
df = pd.read_csv(filename)         # Read the CSV into a DataFrame

# Checkpoint to ensure the file has been loaded 
print(filename, 'has been loaded')


In [ ]:
# import pandas as pd
import pingouin as pg
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

df = phys

# 1. Define the variables in the desired order
vars_ordered = ['Rehearsal', 'Elaboration', 'State_Anxiety', 'Grade', 'SelfEfficacy']

# 2. Create dummy covariates for Gender and Course
covariates = pd.get_dummies(df[['Gender','Course']], drop_first=True)

# 3. Combine numeric columns + covariates
df_numeric = pd.concat([df, covariates], axis=1)

# 4. Create empty DataFrames for r values and annotations
pcorr_matrix = pd.DataFrame(index=vars_ordered, columns=vars_ordered, dtype=float)
annot_matrix = pd.DataFrame(index=vars_ordered, columns=vars_ordered, dtype=str)

# 5. Compute partial correlations and significance
for var1 in vars_ordered:
    for var2 in vars_ordered:
        if var1 == var2:
            pcorr_matrix.loc[var1, var2] = 1.0
            annot_matrix.loc[var1, var2] = '1.00'
        else:
            result = pg.partial_corr(data=df_numeric, x=var1, y=var2, covar=covariates.columns.tolist())
            r = result['r'].values[0]
            p = result['p-val'].values[0]

            # Assign significance stars
            if p < 0.01:
                star = '**'
            elif p < 0.05:
                star = '*'
            else:
                star = ''

            pcorr_matrix.loc[var1, var2] = r
            annot_matrix.loc[var1, var2] = f'{r:.2f}{star}'

# 6. Plot the heatmap
plt.figure(figsize=(10,8))
sns.heatmap(pcorr_matrix.astype(float), annot=annot_matrix, fmt='', cmap='coolwarm', center=0, square=True)
plt.title('Partial Correlation Heatmap (controlling for Gender & Course)')
plt.savefig('PartialCorr_Life')
plt.show()
